# 04. Tendências Temporais e Projeções

**Objetivo:** Analisar tendências de desmatamento e projetar cenários futuros para detecção precoce de padrões anormais.

**Impacto no Negócio:** Identificar municípios com tendências de aumento de desmatamento para alertas precoces e intervenções preventivas.

**Dados de Entrada:** `data/04_modelagem/dataset_preditivo_com_precos.parquet`

**Dados de Saída:** 
- `data/03_gold/tendencias_desmatamento_temporal.parquet`
- `data/03_gold/projecao_desmatamento_2024.parquet`

In [ ]:
# ============================================================================
# MONTAR GOOGLE DRIVE (APENAS COLAB)
# ============================================================================

def montar_google_drive():
    """Monta o Google Drive no Colab."""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✓ Google Drive montado em /content/drive")
        return True
    except Exception as e:
        print(f"⚠️  Erro ao montar Google Drive: {e}")
        return False

# Detecta se está no Colab e tenta montar o Drive
try:
    import google.colab
    print("📤 Ambiente Google Colab detectado")
    print("Montando Google Drive...")
    montar_google_drive()
except ImportError:
    print("✓ Ambiente local detectado - não é necessário montar Drive")

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DE AMBIENTE
# ============================================================================

import sys
import os
from pathlib import Path

# Detectar ambiente e configurar caminho corretamente
try:
    import google.colab
    print("📤 Ambiente Google Colab detectado")
    # No Colab, usar o diretório do drive
    if os.path.exists('/content/drive/MyDrive/dados_analise'):
        os.chdir('/content/drive/MyDrive/dados_analise')
        print("✓ Diretório alterado para: /content/drive/MyDrive/dados_analise")
    else:
        print("⚠️  Diretório dados_analise não encontrado no Drive")
except ImportError:
    print("✓ Ambiente local detectado")
    # Local, usar diretório atual
    current_dir = Path.cwd()
    # Se estiver em notebooks_analise_preditiva, voltar para o root
    if 'notebooks_analise_preditiva' in str(current_dir):
        os.chdir(current_dir.parent)
        print(f"✓ Diretório alterado para: {current_dir.parent}")

print(f"✓ Diretório de trabalho atual: {os.getcwd()}")

# ============================================================================
# CONFIGURAÇÃO DE CAMINHOS
# ============================================================================

# Caminho direto para os dados (já está no drive)
CAMINHO_DADOS = 'data/04_modelagem/dataset_preditivo_com_precos.parquet'
CAMINHO_TENDENCIAS = 'data/03_gold/tendencias_desmatamento_temporal.parquet'

print(f"\nCaminho dos dados: {CAMINHO_DADOS}")
print(f"Caminho tendências: {CAMINHO_TENDENCIAS}")

# Configurar caminho de projeção
if os.path.exists('/content/drive'):
    CAMINHO_PROJECAO = '/content/data/03_gold/projecao_desmatamento_2024.parquet'
else:
    CAMINHO_PROJECAO = 'data/03_gold/projecao_desmatamento_2024.parquet'

print(f"Caminho projeção: {CAMINHO_PROJECAO}")

In [ ]:
## 1. Configuração e Importações
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURAÇÃO
# ============================================================================

# Configuração de thresholds para classificação
THRESHOLDS_TENDENCIA = {
    'reducao_forte': THRESHOLD_REDUCAO_FORTE,
    'reducao_leve': THRESHOLD_REDUCAO_LEVE,
    'aumento_leve': THRESHOLD_AUMENTO_LEVE,
    'aumento_forte': THRESHOLD_AUMENTO_FORTE
}

## 2. Carregamento e Preparação dos Dados

In [ ]:
# ============================================================================
# PASSO 1: CARREGAMENTO E PREPARAÇÃO DOS DADOS
# ============================================================================

# Carregar dataset principal usando função auxiliar
df = carregar_dados(CAMINHO_DADOS)
print('Dataset carregado:', df.shape)
print('Colunas:', df.shape[1])
print('Período:', df['ano'].min(), '-', df['ano'].max())
print('Municípios:', df['cod_ibge'].nunique())

In [ ]:
# ============================================================================
# PASSO 2: FILTRAGEM DA AMAZÔNIA LEGAL
# ============================================================================

# Filtrar Amazônia Legal usando função auxiliar
df_amazonia = filtrar_amazonia_legal(df, UFS_AMAZONIA_LEGAL)
print(f'\nAmazônia Legal: {df_amazonia.shape[0]:,} observações, {df_amazonia["cod_ibge"].nunique():,} municípios')

# Estatísticas gerais
area_total = df_amazonia["area_desmatada_ha"].sum()
print(f'\nÁrea total desmatada: {area_total:,.0f} ha')
print(f'Área desmatada por ano:')
area_por_ano = df_amazonia.groupby('ano')['area_desmatada_ha'].sum()
print(area_por_ano)

In [ ]:
# ============================================================================
# PASSO 3: ANÁLISE DE TENDÊNCIAS POR MUNICÍPIO
# ============================================================================

# Calcular tendências por município usando função auxiliar
tendencias = calcular_tendencias_por_municipio(df_amazonia)

print(f'Tendências calculadas para {len(tendencias)} municípios')
print(f'\nEstatísticas das tendências:')
print(tendencias['tendencia_desmatamento'].describe())

In [ ]:
# ============================================================================
# PASSO 4: CLASSIFICAÇÃO DE TENDÊNCIAS
# ============================================================================

# Definir thresholds de classificação
thresholds_tendencia = {
    'reducao_forte': THRESHOLD_REDUCAO_FORTE,
    'reducao_leve': THRESHOLD_REDUCAO_LEVE,
    'aumento_leve': THRESHOLD_AUMENTO_LEVE,
    'aumento_forte': THRESHOLD_AUMENTO_FORTE
}

# Classificar tendências usando função auxiliar
tendencias = classificar_tendencias_dataframe(tendencias, thresholds_tendencia)

print('\nDistribuição de tendências:')
print(tendencias['categoria_tendencia'].value_counts().sort_index())

In [ ]:
# ============================================================================
# PASSO 5: IDENTIFICAÇÃO DE MUNICÍPIOS CRÍTICOS
# ============================================================================

# Identificar municípios com tendência de aumento forte
municipios_aumento = tendencias[tendencias['categoria_tendencia'] == 'Aumento Forte'].copy()
print(f'\nMunicípios com tendência de aumento forte: {len(municipios_aumento)}')

# Obter informações dos municípios usando função auxiliar
info_municipios = obter_informacoes_municipios(df_amazonia)
municipios_aumento = municipios_aumento.merge(info_municipios, on='cod_ibge')

print('\n=== TOP 20 MUNICÍPIOS COM MAIOR TENDÊNCIA DE AUMENTO DE DESMATAMENTO ===')
top_20_aumento = municipios_aumento.sort_values('tendencia_desmatamento', ascending=False).head(20)
print(top_20_aumento[['cod_ibge', 'municipio', 'uf', 'tendencia_desmatamento', 
                      'area_desmatada_ha', 'vab_agro_mil_reais']].to_string(index=False))

In [ ]:
# ============================================================================
# PASSO 6: ANÁLISE DE SAZONALIDADE
# ============================================================================

# Análise de sazonalidade usando função auxiliar
print('\n=== ANÁLISE DE SAZONALIDADE ===')
sazonalidade = analisar_sazonalidade(df_amazonia)

print('Desmatamento por estação chuvosa:')
print(sazonalidade.pivot(index='ano', columns='estacao_chuva', values='area_total'))

In [ ]:
# ============================================================================
# PASSO 7: PROJEÇÃO PARA 2024
# ============================================================================

# Obter dados do ano atual
df_ano_atual = df_amazonia[df_amazonia['ano'] == ANO_ATUAL].copy()

# Projetar desmatamento para 2024 usando função auxiliar
projecao_2024 = projetar_desmatamento_ano_seguinte(
    df_ano_atual, 
    tendencias, 
    ANO_ATUAL, 
    ANO_PROJECAO
)

# Calcular totais
area_proj_total = projecao_2024['area_desmatada_proj'].sum()
area_atual_total = df_ano_atual['area_desmatada_ha'].sum()

print(f'\n=== PROJEÇÃO DE DESMATAMENTO PARA {ANO_PROJECAO} ===')
print(f'Área desmatada {ANO_ATUAL}: {area_atual_total:,.0f} ha')
print(f'Área projetada {ANO_PROJECAO}: {area_proj_total:,.0f} ha')
print(f'Variação projetada: {((area_proj_total/area_atual_total - 1)*100):.1f}%')

In [ ]:
# ============================================================================
# PASSO 8: TOP MUNICÍPIOS POR PROJEÇÃO
# ============================================================================

# Municípios com maior aumento projetado
projecao_top = projecao_2024.sort_values('area_desmatada_proj', ascending=False).head(20)
print('\n=== TOP 20 MUNICÍPIOS COM MAIOR ÁREA PROJETADA PARA 2024 ===')
print(projecao_top[['cod_ibge', 'municipio', 'uf', 'area_desmatada_ha', 
                     'tendencia_desmatamento', 'area_desmatada_proj']].to_string(index=False))

In [ ]:
# ============================================================================
# PASSO 9: SALVAMENTO DOS RESULTADOS
# ============================================================================

# Salvar tendências
tendencias_completas = tendencias.merge(info_municipios, on='cod_ibge')
salvar_resultado(tendencias_completas, CAMINHO_TENDENCIAS)
print(f'\nTendências salvas em {CAMINHO_TENDENCIAS}')
print(f'Total de municípios: {len(tendencias_completas)}')

# Salvar projeção
projecao_2024['ano_projecao'] = ANO_PROJECAO
salvar_resultado(projecao_2024, CAMINHO_PROJECAO)
print(f'\nProjeção salva em {CAMINHO_PROJECAO}')
print(f'Total de municípios: {len(projecao_2024)}')

## 7. Projeção para 2024

In [ ]:
# Projeção simples para 2024 baseada em tendências
projecao_2024 = df_amazonia[df_amazonia['ano'] == ANO_ATUAL].copy()
projecao_2024 = projecao_2024.merge(tendencias[['cod_ibge', 'tendencia_desmatamento']], on='cod_ibge')

# Aplicar tendência para projetar 2024
projecao_2024['area_desmatada_proj_2024'] = (
    projecao_2024['area_desmatada_ha'] + projecao_2024['tendencia_desmatamento']
)
projecao_2024['area_desmatada_proj_2024'] = projecao_2024['area_desmatada_proj_2024'].clip(lower=0)

# Total projetado
area_proj_total = projecao_2024['area_desmatada_proj_2024'].sum()
area_atual_total = df_amazonia[df_amazonia['ano'] == ANO_ATUAL]['area_desmatada_ha'].sum()

print(f'\n=== PROJEÇÃO DE DESMATAMENTO PARA {ANO_PROJECAO} ===')
print(f'Área desmatada {ANO_ATUAL}: {area_atual_total:,.0f} ha')
print(f'Área projetada {ANO_PROJECAO}: {area_proj_total:,.0f} ha')
print(f'Variação projetada: {((area_proj_total/area_atual_total - 1)*100):.1f}%')

## 8. Top Municípios por Projeção

In [ ]:
# Municípios com maior aumento projetado
projecao_top = projecao_2024.sort_values('area_desmatada_proj_2024', ascending=False).head(20)
print('\n=== TOP 20 MUNICÍPIOS COM MAIOR ÁREA PROJETADA PARA 2024 ===')
print(projecao_top[['cod_ibge', 'municipio', 'uf', 'area_desmatada_ha', 
                     'tendencia_desmatamento', 'area_desmatada_proj_2024']].to_string(index=False))

## 9. Salvamento dos Resultados

In [ ]:
# Salvar tendências
tendencias_completas = tendencias.merge(info_municipios, on='cod_ibge')
tendencias_completas.to_parquet(CAMINHO_TENDENCIAS, index=False)
print(f'\nTendências salvas em {CAMINHO_TENDENCIAS}')
print(f'Total de municípios: {len(tendencias_completas)}')

# Salvar projeção
projecao_2024.to_parquet(CAMINHO_PROJECAO, index=False)
print(f'\nProjeção salva em {CAMINHO_PROJECAO}')
print(f'Total de municípios: {len(projecao_2024)}')

## 10. Conclusão

**Resumo da Análise:**
- Tendências calculadas para {len(tendencias)} municípios
- {len(municipios_aumento)} municípios com tendência de aumento forte
- Projeção de desmatamento para {ANO_PROJECAO}: {area_proj_total:,.0f} ha
- Variação projetada: {((area_proj_total/area_atual_total - 1)*100):.1f}%

**Impacto no Negócio:**
- Detecção precoce de municípios com tendência de aumento
- Base para alertas preditivos e intervenções preventivas
- Planejamento de recursos de fiscalização futuro

**Próximos Passos:**
- Integrar com sistemas de monitoramento satelital em tempo real
- Validar projeções com dados de {ANO_PROJECAO} quando disponíveis
- Implementar alertas automáticos para municípios com tendência de aumento forte